# Venn (interactive)

> The same area-proportional Euler layout as [Venn (static)](venn.html), rendered in Plotly with members shown on hover.

In [1]:
#| default_exp venn_interactive

In [2]:
#| export
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from itertools import combinations
import colorsys
import plotly.graph_objects as go
import numpy as np
import eunoia as eu

In [3]:
#| export
def _sat(rgb):
    h, s, v = colorsys.rgb_to_hsv(*rgb)
    return colorsys.hsv_to_rgb(h, min(1.0, s * 1.8), v * 0.75)

def _pack(items, max_rows, gutter=3):
    cols = max(1, -(-len(items) // max_rows))
    rows = -(-len(items) // cols)
    col_items = [items[c * rows:(c + 1) * rows] for c in range(cols)]
    widths = [max((len(i) for i in col), default=0) + gutter for col in col_items]
    lines = []
    for r in range(rows):
        line = "".join(col_items[c][r].ljust(widths[c]) if r < len(col_items[c]) else " " * widths[c] for c in range(cols))
        lines.append(line.rstrip().replace(" ", " "))
    return "<br>".join(lines)

def _outer_pos(outlines, labels, pad):
    "Position each set name just outside its shape, offset radially from the diagram centre."
    cents = {lab: np.asarray(outlines[lab]).mean(0) for lab in labels}
    C0 = np.concatenate([np.asarray(outlines[lab]) for lab in labels]).mean(0)
    out = {}
    for idx, lab in enumerate(labels):
        pts = np.asarray(outlines[lab])
        d = cents[lab] - C0
        if np.hypot(*d) < 1e-9:
            a = np.pi / 2 - idx * 2 * np.pi / len(labels)
            d = np.array([np.cos(a), np.sin(a)])
        d = d / np.hypot(*d)
        far = pts[(pts @ d).argmax()]
        p = far + d * pad
        xa = "left" if d[0] > 0.3 else "right" if d[0] < -0.3 else "center"
        ya = "bottom" if d[1] > 0.3 else "top" if d[1] < -0.3 else "middle"
        out[lab] = (float(p[0]), float(p[1]), xa, ya)
    return out

def eunoia_venn_interactive(dfs, colors, style="round", title="", opacity=0.5, count_size=14,
                            count_color=None, title_size=20, name_size=16, width=760, height=680,
                            show_counts=True, hover_size=13, hit_size=34, save_path=None):
    # style: "round" -> circle (2 sets) / ellipse (3 sets); "box" -> square (2 sets) / rectangle (3 sets)
    # hover any region to list its members; counts are shown in place and set names sit outside each shape
    labels = [d.columns[0] for d in dfs]
    lists = [d.iloc[:, 0].tolist() for d in dfs]
    sets = [set(l) for l in lists]
    n = len(dfs)
    if isinstance(colors, str):
        cmap = plt.get_cmap(colors)
        pal = cmap.colors if hasattr(cmap, "colors") else [cmap(k / max(n - 1, 1)) for k in range(n)]
        colors = [pal[k % len(pal)] for k in range(n)]
    rgb = [mpl.colors.to_rgb(c) for c in colors]
    shape = {"round": "ellipse" if n == 3 else "circle", "box": "rectangle" if n == 3 else "square"}[style]
    fit = eu.euler({labels[i]: lists[i] for i in range(n)}, shape=shape)
    rp = fit.plot_data["region_pieces"]
    outlines = fit.plot_data["shape_outlines"]
    anchors = fit.plot_data["region_anchors"]

    fig = go.Figure()
    figure_rows = (height - 180) / (hover_size * 1.45)
    allx = [p[0] for o in outlines.values() for p in o]
    ally = [p[1] for o in outlines.values() for p in o]
    total_h = max(ally) - min(ally)
    span = max(max(allx) - min(allx), total_h)
    markers = []
    for r in range(1, n + 1):
        for combo in combinations(range(n), r):
            key = "&".join(sorted(labels[i] for i in combo))
            if key not in rp:
                continue
            s = set.intersection(*(sets[i] for i in combo)).difference(*(sets[i] for i in range(n) if i not in combo))
            if not s:
                continue
            items = [p for p in lists[combo[0]] if p in s]
            parts = [rgb[i] for i in combo]
            m = [sum(c) / len(parts) for c in zip(*parts)]
            R, G, B = int(m[0] * 255), int(m[1] * 255), int(m[2] * 255)
            for piece in rp[key]:
                xs = [p[0] for p in piece["outer"]]
                ys = [p[1] for p in piece["outer"]]
                fig.add_trace(go.Scatter(x=xs, y=ys, fill="toself", fillcolor=f"rgba({R},{G},{B},{opacity})",
                                         line=dict(width=0), mode="lines", hoverinfo="skip", showlegend=False))
            ys_all = [p[1] for piece in rp[key] for p in piece["outer"]]
            budget = max(3, min(int(figure_rows), round(figure_rows * (max(ys_all) - min(ys_all)) / total_h) + 5))
            hov = _pack(items, budget)
            if key in anchors:
                cc = count_color if count_color is not None else mpl.colors.to_hex(_sat(m))
                markers.append((anchors[key][0], anchors[key][1], len(s), hov, cc))
    for i, lab in enumerate(labels):
        o = outlines[lab]
        xs = [p[0] for p in o] + [o[0][0]]
        ys = [p[1] for p in o] + [o[0][1]]
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=mpl.colors.to_hex(rgb[i]), width=1.5),
                                 hoverinfo="skip", showlegend=False))
    for ax_, ay_, cnt, hov, col in markers:
        fig.add_trace(go.Scatter(x=[ax_], y=[ay_], mode="markers", marker=dict(size=hit_size, color="rgba(0,0,0,0)"),
                                 hovertemplate=hov + "<extra></extra>",
                                 hoverlabel=dict(bgcolor="rgba(211,211,211,0.6)", bordercolor="rgba(120,120,120,0.6)",
                                                 font=dict(size=hover_size, color="black", family="Inter, sans-serif")),
                                 showlegend=False))
        if show_counts:
            fig.add_annotation(x=ax_, y=ay_, text=str(cnt), showarrow=False,
                               font=dict(size=count_size, color=col, family="Inter, sans-serif"))
    pos = _outer_pos(outlines, labels, 0.03 * span)
    for i, lab in enumerate(labels):
        x, y, xa, ya = pos[lab]
        fig.add_annotation(x=x, y=y, text=f"<b>{lab}</b>", showarrow=False, xanchor=xa, yanchor=ya,
                           font=dict(size=name_size, color=mpl.colors.to_hex(rgb[i]), family="Inter, sans-serif"))
    fig.add_trace(go.Scatter(x=[pos[l][0] for l in labels], y=[pos[l][1] for l in labels],
                             mode="markers", marker=dict(size=1, color="rgba(0,0,0,0)"), hoverinfo="skip", showlegend=False))
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False, scaleanchor="x", scaleratio=1)
    fig.update_layout(title=dict(text=title, font=dict(size=title_size, family="Inter, sans-serif")),
                      width=width, height=height, plot_bgcolor="white", showlegend=False,
                      margin=dict(l=70, r=70, t=90, b=70), hoverlabel=dict(align="left"))
    if save_path:
        fig.write_html(save_path)
    fig.show()

In [4]:
#| hide
from viset.core import load
from matplotlib import font_manager
import matplotlib.pyplot as plt
for f in font_manager.findSystemFonts(fontpaths=["Font" if __import__("os").path.isdir("Font") else "../Font"]):
    font_manager.fontManager.addfont(f)
plt.rcParams["font.family"] = "Inter"

In [5]:
#| hide
import plotly.io as pio
pio.renderers.default = "notebook_connected"

## Data

In [6]:
drugbase = ("Drug target sample files/" if __import__("os").path.isdir("Drug target sample files") else "../Drug target sample files/")
flu = load(drugbase + "1. FLUOXETINE_sample.csv", "Gene names", "Fluoxetine")
ibu = load(drugbase + "2. IBUPROFEN_sample.csv", "Gene names", "Ibuprofen")
acet = load(drugbase + "3. ACETAMINOPHEN_sample.csv", "Gene names", "Acetaminophen")

## Interactive Venn / Euler

Hover any region to list its members; counts and single-set names are drawn in place.

In [7]:
eunoia_venn_interactive([ibu, acet], colors="Set1", width=740, height=560, title="Ibuprofen vs Acetaminophen targets")

In [8]:
eunoia_venn_interactive([flu, ibu, acet], colors="Set1", width=760, height=640, title="Targets of Fluoxetine, Ibuprofen, Acetaminophen")

## Controlling plot aesthetics

| Argument | Controls |
|---|---|
| `style` | `"round"` or `"box"`, as in the static version |
| `colors` | list of colours, or a colormap name |
| `title`, `title_size` | figure title text and size |
| `count_size` | font size of the in-region **count** numbers |
| `count_color` | colour of the counts; pass e.g. `"black"` to override the default darkened blend |
| `name_size` | font size of the **set names** outside each shape |
| `opacity` | fill opacity of the shapes |
| `hover_size` | font size of the member list in the hover tooltip |
| `show_counts` | draw the counts in place (`True`) or leave members to hover only |
| `width`, `height` | figure size in pixels |
| `save_path` | optional path to save a standalone HTML file |

**Colormap names.** Any Matplotlib colormap name works for `colors`; qualitative maps keep the sets/bars most distinct: `Set1`, `Set2`, `Set3`, `Dark2`, `Paired`, `Accent`, `Pastel1`, `Pastel2`, `tab10`, `tab20`. You can also pass an explicit list of colours instead.